In [1]:
import random
from typing import List, Dict
from langchain_community.chat_models import ChatOllama
from langchain.agents import initialize_agent, AgentType
from persona import generate_persona, Persona  # 參考您提供的 persona.py

# === 1. 實驗參數設定 (依據論文實驗設計) ===
NUM_AGENTS = 5  # 模擬的代理人數量
SIMULATION_ROUNDS = 3  # 模擬對話的輪數
TOPIC = "關於數位身分證(eID)推行的隱私與便利性爭議" # 論文常見的社會議題範例

# === 2. 初始化 LLM 與角色 (Persona) ===
# 論文中使用不同的 LLM 來模擬多樣性
llm_pool = [
    ChatOllama(model="llama3:8B"),
    ChatOllama(model="mistral:7B"),
    ChatOllama(model="gemma3:4b")
]

def initialize_social_simulation():
    agents = {}
    personas = {}
    
    # 生成具備初始立場的角色
    # 論文中會隨機分配支持、反對、中立立場
    stances = ["支持", "反對", "中立"]
    
    for i in range(NUM_AGENTS):
        name = f"Agent_{i+1}"
        initial_stance = random.choice(stances)
        
        # 使用 persona.py 的生成功能
        p = generate_persona(name=name, initial_stance=initial_stance)
        personas[name] = p
        
        # 綁定 LLM
        selected_llm = random.choice(llm_pool)
        
        # 建立 Agent
        agents[name] = initialize_agent(
            tools=[], # 論文實驗通常聚焦於對話，不一定使用外部工具
            llm=selected_llm,
            handle_parsing_errors=True,
            agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
            verbose=False,
            agent_kwargs={
                "prefix": p.to_prompt(), # 注入論文核心的 Persona 描述
            }
        )
    return agents, personas

# === 3. 社會模擬流程 (模擬論文中的對話機制) ===
def run_social_influence_simulation():
    agents, personas = initialize_social_simulation()
    conversation_history = []
    
    print(f"=== 模擬開始：關於 {TOPIC} 的討論 ===")
    
    for r in range(SIMULATION_ROUNDS):
        print(f"\n--- 第 {r+1} 輪對話 ---")
        
        # 論文中通常採用隨機輪詢或順序輪詢
        agent_names = list(agents.keys())
        random.shuffle(agent_names)
        
        for name in agent_names:
            # 建立 Context：包含之前的對話摘要（論文中的 Short-term Memory）
            context = "\n".join(conversation_history[-5:]) # 取最近5條記錄
            
            prompt = f"""
            目前的討論話題是：{TOPIC}。
            這是目前的討論進度：
            {context}
            
            請根據你的角色人格(Persona)與立場，對目前的討論做出回應。
            你不需要表現得像 AI，請像一個真實的人類在社交平台發言。
            """
            
            response = agents[name].run(input=prompt, chat_history=[])
            
            formatted_entry = f"{name} ({personas[name].initial_stance}): {response}"
            conversation_history.append(formatted_entry)
            print(formatted_entry)

    # === 4. 立場變化分析 (論文核心分析點) ===
    analyze_opinion_shift(agents, personas, conversation_history)

def analyze_opinion_shift(agents, personas, history):
    print("\n=== 實驗結果：意見偏好分析 ===")
    # 論文會透過 LLM 作為評審 (LLM-as-a-judge) 來判斷代理人對話後的立場是否改變
    analyzer = ChatOllama(model="llama3:8B")
    
    for name, persona in personas.items():
        analysis_prompt = f"""
        角色：{persona.name}
        初始立場：{persona.initial_stance}
        
        以下是他在討論中的發言：
        {[h for h in history if h.startswith(name)]}
        
        請判斷他在討論結束後，立場是否有發生「從眾效應」或「意見極化」？
        請簡短回答。
        """
        result = analyzer.invoke(analysis_prompt)
        print(f"[{name}] 立場分析：{result.content}")

if __name__ == "__main__":
    run_social_influence_simulation()

C:\Users\TUF\AppData\Local\Temp\ipykernel_25536\1715336326.py:15: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  ChatOllama(model="llama3:8B"),
C:\Users\TUF\AppData\Local\Temp\ipykernel_25536\1715336326.py:40: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.lang

=== 模擬開始：關於 關於數位身分證(eID)推行的隱私與便利性爭議 的討論 ===

--- 第 1 輪對話 ---
Agent_5 (反對): 你說的完全沒錯！數位身分證的討論確實要從隱私和便利性兩個方面著手。我認為，像是資料加密、嚴格審核制度這種措施，對於保護個人隱私至關重要。而且，讓民眾充分了解數位身分證的運作方式，也能夠讓大家更安心使用。要達成隱私與便利性兼顧，真的需要政府、民眾之間多一點的溝通和協作。
Agent_3 (反對): Agent stopped due to iteration limit or time limit.
Agent_4 (中立): I completely agree with you both! The discussion on eID should indeed focus on privacy and convenience. I believe that measures such as data encryption, strict review systems, and transparency in the operation of eID can be crucial for protecting personal privacy. Additionally, educating the public about how eID works can help build trust among users. To strike a balance between privacy and convenience, it is indeed important for government and the public to engage in more communication and collaboration.
Agent_1 (反對): Agent stopped due to iteration limit or time limit.
Agent_2 (中立): Let's focus on finding a balance between privacy and convenience in the implementation of eID.

--- 第 2 輪對話 ---
Agent_5